# GOV-01 V2 Multi-label E2: threshold calibration recovery

This notebook loads the already-trained E2 model from Google Drive and uses validation data only. It does not train, change the model, or load the protected test set.

In [ ]:
# Connect Google Drive and set the two required file locations.
from pathlib import Path
import csv
import json
import shutil
import zipfile

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/GOV-01')
DRIVE_DATASET_ZIP = DRIVE_ROOT / 'v2_multilabel.zip'
DRIVE_MODEL = DRIVE_ROOT / 'v2_multilabel_e2_finetuned_best.keras'
LOCAL_DATA_ROOT = Path('/content/v2_multilabel')
OUTPUT_DIRECTORY = Path('/content/v2_multilabel_e2_calibration_output')
CONDITIONS = ('crack', 'pothole', 'repaired_road', 'manhole_cover', 'unpaved_road', 'road_marking', 'speed_bump')

if not DRIVE_DATASET_ZIP.is_file():
    raise FileNotFoundError(f'Dataset ZIP not found: {DRIVE_DATASET_ZIP}')
if not DRIVE_MODEL.is_file():
    raise FileNotFoundError(f'Saved E2 model not found: {DRIVE_MODEL}')

print('Dataset ZIP:', DRIVE_DATASET_ZIP)
print('Saved E2 model:', DRIVE_MODEL)
print('This notebook uses validation only; it does not load the protected test set.')


In [ ]:
# Extract the dataset only when the required validation files are not already complete.
def validation_is_complete(dataset_root):
    split_root = dataset_root / 'validation'
    labels_path = split_root / 'labels.csv'
    if not labels_path.is_file() or not (split_root / 'images').is_dir():
        return False
    with labels_path.open(newline='', encoding='utf-8') as source:
        for row in csv.DictReader(source):
            relative_image = Path(row['materialized_image'].replace(chr(92), '/'))
            if not (split_root / relative_image).is_file():
                return False
    return True

if validation_is_complete(LOCAL_DATA_ROOT):
    print('Using verified temporary validation data:', LOCAL_DATA_ROOT)
else:
    if LOCAL_DATA_ROOT.exists():
        print('Removing incomplete temporary extraction:', LOCAL_DATA_ROOT)
        shutil.rmtree(LOCAL_DATA_ROOT)
    print('Extracting dataset ZIP to temporary Colab storage. This can take several minutes.')
    with zipfile.ZipFile(DRIVE_DATASET_ZIP) as archive:
        archive.extractall(LOCAL_DATA_ROOT.parent)
    if not validation_is_complete(LOCAL_DATA_ROOT):
        raise RuntimeError('The ZIP was extracted, but validation files are incomplete. Check that the Drive ZIP is v2_multilabel.zip, then run this cell again.')
    print('Validation extraction verified:', LOCAL_DATA_ROOT)

OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)


In [ ]:
# Load validation labels and make predictions. A GPU is optional for this recovery task.
import numpy as np
import tensorflow as tf
from sklearn.metrics import f1_score

print('TensorFlow:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))
print('No GPU is required here because this notebook makes predictions only; it does not train.')

def read_validation():
    split_root = LOCAL_DATA_ROOT / 'validation'
    paths, labels, masks = [], [], []
    with (split_root / 'labels.csv').open(newline='', encoding='utf-8') as source:
        for row in csv.DictReader(source):
            relative_image = Path(row['materialized_image'].replace(chr(92), '/'))
            image_path = split_root / relative_image
            if not image_path.is_file():
                raise FileNotFoundError(f'Missing validation image: {image_path}')
            label_row, mask_row = [], []
            for condition in CONDITIONS:
                known = int(row[f'{condition}_known'])
                value = row[f'{condition}_present']
                if known == 0 and value != '':
                    raise ValueError(f'Unknown label has a value for {condition}: {row["record_id"]}')
                label_row.append(float(value) if known else 0.0)
                mask_row.append(float(known))
            paths.append(str(image_path))
            labels.append(label_row)
            masks.append(mask_row)
    return np.asarray(paths), np.asarray(labels, dtype='float32'), np.asarray(masks, dtype='float32')

validation_paths, validation_labels, validation_masks = read_validation()
print('Validation images:', len(validation_paths))
for index, condition in enumerate(CONDITIONS):
    print(f'{condition:16} known={int(validation_masks[:, index].sum()):4}  positive={int((validation_labels[:, index] * validation_masks[:, index]).sum()):4}')

def load_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, (224, 224))
    return tf.cast(image, tf.float32)

validation_images = tf.data.Dataset.from_tensor_slices(validation_paths)
validation_images = validation_images.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
validation_images = validation_images.batch(32).prefetch(tf.data.AUTOTUNE)

best_model = tf.keras.models.load_model(DRIVE_MODEL, compile=False)
validation_probabilities = best_model.predict(validation_images, verbose=1)
print('Prediction shape:', validation_probabilities.shape)


In [ ]:
# Select one F1-maximizing threshold for each road condition using validation data only.
import matplotlib.pyplot as plt

THRESHOLD_GRID = np.round(np.arange(0.20, 0.81, 0.05), 2)
threshold_calibration = {}
figure, axes = plt.subplots(2, 4, figsize=(14, 7))

for index, condition in enumerate(CONDITIONS):
    known = validation_masks[:, index].astype(bool)
    actual = validation_labels[known, index].astype(int)
    probabilities = validation_probabilities[known, index]
    f1_scores = [float(f1_score(actual, probabilities >= threshold, zero_division=0)) for threshold in THRESHOLD_GRID]
    best_index = max(range(len(THRESHOLD_GRID)), key=lambda item: (f1_scores[item], -abs(float(THRESHOLD_GRID[item]) - 0.50)))
    default_index = int(np.where(np.isclose(THRESHOLD_GRID, 0.50))[0][0])
    threshold_calibration[condition] = {
        'known_support': int(len(actual)),
        'positive_support': int(actual.sum()),
        'default_threshold': 0.50,
        'default_f1': f1_scores[default_index],
        'selected_threshold': float(THRESHOLD_GRID[best_index]),
        'selected_f1': f1_scores[best_index],
        'f1_improvement': f1_scores[best_index] - f1_scores[default_index],
    }
    axis = axes.flat[index]
    axis.plot(THRESHOLD_GRID, f1_scores, marker='o')
    axis.axvline(0.50, color='gray', linestyle='--', label='current')
    axis.axvline(THRESHOLD_GRID[best_index], color='green', linestyle='--', label='selected')
    axis.set_title(condition)
    axis.set_xlabel('threshold')
    axis.set_ylabel('validation F1')
    axis.legend(fontsize=8)

axes.flat[-1].axis('off')
figure.tight_layout()
figure.savefig(OUTPUT_DIRECTORY / 'v2_multilabel_e2_threshold_calibration.png', dpi=150)

calibration_json = OUTPUT_DIRECTORY / 'v2_multilabel_e2_threshold_calibration.json'
calibration_csv = OUTPUT_DIRECTORY / 'v2_multilabel_e2_threshold_calibration.csv'
calibration_json.write_text(json.dumps(threshold_calibration, indent=2) + '\n')
with calibration_csv.open('w', newline='', encoding='utf-8') as destination:
    writer = csv.DictWriter(destination, fieldnames=['condition', *next(iter(threshold_calibration.values())).keys()])
    writer.writeheader()
    for condition, values in threshold_calibration.items():
        writer.writerow({'condition': condition, **values})

for condition, values in threshold_calibration.items():
    print(f'{condition:16} threshold={values["selected_threshold"]:.2f}  F1: {values["default_f1"]:.3f} -> {values["selected_f1"]:.3f}  change={values["f1_improvement"]:+.3f}')

print('Saved:', calibration_json)
print('Protected test remains untouched.')


In [ ]:
# Download the calibration result. Keep this ZIP outside Git.
from google.colab import files

result_zip = Path('/content/v2_multilabel_e2_threshold_calibration_output.zip')
if result_zip.exists():
    result_zip.unlink()
shutil.make_archive(str(result_zip.with_suffix('')), 'zip', OUTPUT_DIRECTORY)
print('Downloading:', result_zip.name)
files.download(str(result_zip))
